In [1]:
import os
import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import subprocess
import urllib.request

# -------------------------------------------------------------
# 1. FILE PATH DETECTION
# -------------------------------------------------------------
def find_file(name):
    for path in [f"/content/{name}", f"/{name}", name]:
        if os.path.exists(path):
            return path
    return name

BG_IMAGE = find_file("QuestionPPT.pptx.jpg")
AUDIO_FILE = find_file("Audio.mpeg")
TEMP_VIDEO = "/content/temp_video.mp4"
FINAL_VIDEO = "/content/PW_Pen_Stroke_Solution.mp4"

FPS = 30
DURATION = 71.0
TOTAL_FRAMES = int(DURATION * FPS)
WIDTH, HEIGHT = 1280, 720

# -------------------------------------------------------------
# 2. OLD STYLUS HANDWRITING FONT (Gochi Hand)
# -------------------------------------------------------------
FONT_URL = "https://github.com/google/fonts/raw/main/ofl/gochihand/GochiHand-Regular.ttf"
FONT_PATH = "/content/GochiHand-Regular.ttf"
if not os.path.exists(FONT_PATH):
    try:
        urllib.request.urlretrieve(FONT_URL, FONT_PATH)
    except Exception:
        FONT_PATH = None

if FONT_PATH and os.path.exists(FONT_PATH):
    font_formula = ImageFont.truetype(FONT_PATH, 34)
    font_step    = ImageFont.truetype(FONT_PATH, 32)
    font_result  = ImageFont.truetype(FONT_PATH, 36)
else:
    sys_font = "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"
    if not os.path.exists(sys_font):
        sys_font = "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf"
    font_formula = ImageFont.truetype(sys_font, 24)
    font_step    = ImageFont.truetype(sys_font, 23)
    font_result  = ImageFont.truetype(sys_font, 26)

INK_BLACK = (30, 30, 35)
INK_BLUE  = (20, 75, 205)
INK_GREEN = (25, 140, 45)
INK_RED   = (215, 35, 35)
PEN_TIP   = (235, 45, 45)

# -------------------------------------------------------------
# 3. AUDIO-ALIGNED TIMELINE
# -------------------------------------------------------------
EVENTS = [
    {"type": "line", "start": 5.4, "end": 71.0, "p1": (512, 108), "p2": (585, 108), "color": INK_RED, "width": 3},
    {"type": "line", "start": 7.0, "end": 71.0, "p1": (630, 108), "p2": (698, 108), "color": INK_RED, "width": 3},

    {"type": "text", "start": 13.5, "end": 21.0, "x": 450, "y": 175, "text": "Distance (d) = [ (x2 - x1)² + (y2 - y1)² ]", "color": INK_BLACK, "font": font_formula},
    {"type": "box",  "start": 21.2, "end": 71.0, "p1": (430, 162), "p2": (1075, 222), "color": INK_BLACK, "width": 2},

    {"type": "text", "start": 30.5, "end": 40.0, "x": 550, "y": 255, "text": "= [ (4 - 1)² + (6 - 2)² ]", "color": INK_BLUE, "font": font_step},
    {"type": "text", "start": 44.2, "end": 49.5, "x": 550, "y": 320, "text": "= [ 3² + 4² ]", "color": INK_BLUE, "font": font_step},
    {"type": "text", "start": 51.2, "end": 54.8, "x": 550, "y": 385, "text": "= [ 9 + 16 ]", "color": INK_BLUE, "font": font_step},
    {"type": "text", "start": 56.2, "end": 58.8, "x": 550, "y": 450, "text": "= 25", "color": INK_BLUE, "font": font_step},

    {"type": "text", "start": 59.8, "end": 62.2, "x": 550, "y": 525, "text": "d = 5 units", "color": INK_GREEN, "font": font_result},
    {"type": "box",  "start": 62.5, "end": 71.0, "p1": (530, 510), "p2": (730, 568), "color": INK_GREEN, "width": 3},

    {"type": "tick", "start": 64.5, "end": 71.0, "base": (175, 288), "color": INK_GREEN, "width": 4}
]

def draw_tick(draw, base, color, width=4):
    x, y = base
    draw.line([(x, y), (x + 8, y + 14)], fill=color, width=width)
    draw.line([(x + 8, y + 14), (x + 24, y - 16)], fill=color, width=width)

# -------------------------------------------------------------
# 4. FRAME RENDERING WITH ENLARGED PEN NIB DOT
# -------------------------------------------------------------
slide_img = Image.open(BG_IMAGE).convert("RGBA").resize((WIDTH, HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
video_writer = cv2.VideoWriter(TEMP_VIDEO, fourcc, FPS, (WIDTH, HEIGHT))

for frame_idx in range(TOTAL_FRAMES):
    current_time = frame_idx / FPS
    current_frame = slide_img.copy()
    draw = ImageDraw.Draw(current_frame)

    for item in EVENTS:
        start_t = item["start"]
        end_t   = item["end"]
        action  = item["type"]

        if current_time >= start_t:
            if action == "line" and current_time <= end_t:
                progress = min(1.0, (current_time - start_t) / 0.35)
                end_x = int(item["p1"][0] + (item["p2"][0] - item["p1"][0]) * progress)
                draw.line([item["p1"], (end_x, item["p1"][1])], fill=item["color"], width=item["width"])

                # Slightly larger pen dot during underline
                if progress < 1.0:
                    tip_x, tip_y = end_x, item["p1"][1]
                    draw.ellipse([tip_x - 5, tip_y - 5, tip_x + 5, tip_y + 5], fill=PEN_TIP)

            elif action == "tick" and current_time <= end_t:
                draw_tick(draw, item["base"], item["color"], item["width"])

            elif action == "text":
                full_text = item["text"]
                if current_time >= end_t:
                    visible_text = full_text
                    draw.text((item["x"], item["y"]), visible_text, font=item["font"], fill=item["color"])
                else:
                    ratio = (current_time - start_t) / (end_t - start_t)
                    char_count = int(ratio * len(full_text))
                    visible_text = full_text[:char_count]
                    draw.text((item["x"], item["y"]), visible_text, font=item["font"], fill=item["color"])

                    # Enlarged Red Pen Nib Dot (radius 6px)
                    if char_count > 0:
                        bbox = draw.textbbox((item["x"], item["y"]), visible_text, font=item["font"])
                        tip_x = bbox[2] + 5
                        tip_y = item["y"] + (bbox[3] - bbox[1]) // 2
                    else:
                        tip_x = item["x"] + 2
                        tip_y = item["y"] + 14

                    draw.ellipse([tip_x - 6, tip_y - 6, tip_x + 6, tip_y + 6], fill=PEN_TIP)

            elif action == "box" and current_time <= end_t:
                draw.rectangle([item["p1"], item["p2"]], outline=item["color"], width=item["width"])

    bgr_frame = cv2.cvtColor(np.array(current_frame.convert("RGB")), cv2.COLOR_RGB2BGR)
    video_writer.write(bgr_frame)

video_writer.release()

# -------------------------------------------------------------
# 5. AUDIO-VIDEO MERGE
# -------------------------------------------------------------
cmd = [
    "ffmpeg", "-y",
    "-i", TEMP_VIDEO,
    "-i", AUDIO_FILE,
    "-c:v", "libx264",
    "-pix_fmt", "yuv420p",
    "-c:a", "aac",
    "-shortest",
    FINAL_VIDEO
]
subprocess.run(cmd, check=True)

if os.path.exists(TEMP_VIDEO):
    os.remove(TEMP_VIDEO)

try:
    from google.colab import files
    files.download(FINAL_VIDEO)
except Exception:
    pass

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>